<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/AGENTIC_TOPO_PLASTICITY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Thu Oct  1 00:46:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
!pip install -q unsloth

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["UNSLOTH_DISABLE_LOGGING"] = "1"
os.environ["UNSLOTH_DISABLE_ZOO"] = "1"

import io
import sys
import gc
import json
import random
import logging
import warnings
import contextlib
import urllib.request
import numpy as np

warnings.filterwarnings("ignore")

@contextlib.contextmanager
def quiet():
    buf, old = io.StringIO(), (sys.stdout, sys.stderr)
    logging.disable(logging.CRITICAL)
    sys.stdout = sys.stderr = buf
    try:
        yield
    except BaseException:
        sys.stdout, sys.stderr = old
        print(buf.getvalue(), file=sys.stderr)
        raise
    finally:
        sys.stdout, sys.stderr = old
        logging.disable(logging.NOTSET)

with quiet():
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import hashlib
    from PIL import Image
    from huggingface_hub import hf_hub_download
    from safetensors.torch import load_file
    from unsloth import FastVisionModel
    from transformers import AutoModelForCausalLM, AutoTokenizer

# ==============================================================================
# DETERMINISTIC SEED ENFORCEMENT (SEED 123)
# ==============================================================================
SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Initializing Four-Agent Deliberation Council on: {DEVICE}")

# ==============================================================================
# 1. VISION PERCEPTION AGENT (frankmorales2020/topo-gemma-4-e4b-vision-14tasks)
# ==============================================================================
print("\n" + "=" * 90)
print("[1/4] Loading Topo-Gemma Vision Model (14 Tasks via Unsloth)...")
print("=" * 90)

VISION_REPO = "frankmorales2020/topo-gemma-4-e4b-vision-14tasks"
BASE_VISION_MODEL = "frankmorales2020/gemma-4-e4b-unesco-optimized"

LABELS_VISION = {
    "A": ("animal", "vehicle"), "B": ("natural", "man-made"), "C": ("living", "non-living"),
    "D": ("large", "small"), "E": ("ground", "air/water"), "F": ("domestic", "wild"),
    "G": ("mammal", "non-mammal"), "H": ("flying", "non-flying"), "I": ("fast", "slow"),
    "J": ("urban", "rural"), "K": ("predator", "prey"), "L": ("nocturnal", "diurnal"),
    "M": ("domesticated", "wild animal"), "14": ("benign", "malignant"),
}

with quiet():
    vis_ckpt = torch.load(hf_hub_download(VISION_REPO, "pytorch_model.bin"), map_location="cpu", weights_only=False)
    vision_model, vision_processor = FastVisionModel.from_pretrained(
        BASE_VISION_MODEL,
        load_in_4bit=True,
        dtype=torch.bfloat16
    )
    FastVisionModel.for_inference(vision_model)

    emb = vision_model.get_input_embeddings()
    with torch.no_grad():
        emb.weight.copy_(vis_ckpt["embed_tokens_weight"].to(emb.weight.device, emb.weight.dtype))

    vision_heads = {t: nn.Linear(vis_ckpt["hidden_size"], 2).to(DEVICE).eval() for t in vis_ckpt["task_order"]}
    for t, h in vision_heads.items():
        h.load_state_dict(vis_ckpt["classifiers"][t])

vision_tok = getattr(vision_processor, "tokenizer", vision_processor)
print(f"  [✓] Vision Agent Online: Tasks {', '.join(vis_ckpt['task_order'])}")

@torch.no_grad()
def _vision_features(inputs):
    hs = vision_model(**inputs, output_hidden_states=True, use_cache=False).hidden_states[vis_ckpt["boundary_layer"]].float()
    m = inputs["attention_mask"].unsqueeze(-1).float()
    return (hs * m).sum(1) / m.sum(1)

@torch.no_grad()
def predict_vision(task, image=None, text=None):
    task = str(task)
    if task == "14":
        img = Image.open(image).convert("RGB") if isinstance(image, str) else image.convert("RGB")
        msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": vis_ckpt["task14"]["prompt"]}]}]
        prompt = vision_processor.apply_chat_template(msgs, add_generation_prompt=True)
        inputs = vision_processor(img, prompt, add_special_tokens=False, return_tensors="pt").to(DEVICE)
    else:
        enc = vision_tok(text, max_length=64, padding="max_length", truncation=True, return_tensors="pt")
        inputs = {"input_ids": enc["input_ids"].to(DEVICE), "attention_mask": enc["attention_mask"].to(DEVICE)}

    feats = _vision_features(inputs)
    p = torch.softmax(vision_heads[task](feats), -1)[0].tolist()
    k = int(p[1] > p[0])
    return {
        "task": task,
        "label": LABELS_VISION[task][k],
        "confidence": round(p[k], 4),
        "probabilities": {LABELS_VISION[task][0]: round(p[0], 4), LABELS_VISION[task][1]: round(p[1], 4)}
    }

# ==============================================================================
# 2. EMBODIED WORLD MODEL AGENT (frankmorales2020/topo-cbp-world-model-3d)
# ==============================================================================
print("\n" + "=" * 90)
print("[2/4] Loading Topo-CBP Embodied World Model (3D Latent Dynamics)...")
print("=" * 90)

WM_REPO = "frankmorales2020/topo-cbp-world-model-3d"
wm_cfg_file = hf_hub_download(repo_id=WM_REPO, filename="config.json")
wm_weights_file = hf_hub_download(repo_id=WM_REPO, filename="model.safetensors")

with open(wm_cfg_file, "r") as f:
    wm_cfg = json.load(f)

class EmbodiedWorldModel(nn.Module):
    def __init__(self, latent_dim: int = 128, action_dim: int = 6, hidden_dim: int = 256):
        super().__init__()
        self.latent_dim = latent_dim
        self.action_dim = action_dim
        self.hidden_dim = hidden_dim
        self.in_proj = nn.Linear(latent_dim + action_dim, hidden_dim, bias=False)
        self.plastic_linear1 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.plastic_linear2 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj = nn.Linear(hidden_dim, latent_dim, bias=False)

    def forward(self, z_t: torch.Tensor, a_t: torch.Tensor) -> torch.Tensor:
        x = torch.cat([z_t, a_t], dim=-1)
        h = F.silu(self.in_proj(x))
        h1 = self.plastic_linear1(h)
        h2 = F.silu(self.plastic_linear2(h))
        return self.out_proj(h1 * h2)

world_model = EmbodiedWorldModel(wm_cfg["latent_dim"], wm_cfg["action_dim"], wm_cfg["hidden_dim"]).to(DEVICE)
world_model.load_state_dict(load_file(wm_weights_file, device=str(DEVICE)))
world_model.eval()

with torch.no_grad():
    anchors = world_model.in_proj.weight[torch.tensor(wm_cfg["prime_anchors"], dtype=torch.long), :]
    computed_hash = hashlib.sha256(anchors.detach().cpu().numpy().tobytes()).hexdigest()[:16]

interlock_status = "LOCKED" if computed_hash == wm_cfg["invariant_sha256_hash"] else "CORRUPTED"
print(f"  [✓] World Model Online | Target Hash: {wm_cfg['invariant_sha256_hash']} | Interlock: {interlock_status}")

# ==============================================================================
# 3. ALIGNMENT GOVERNOR AGENT (frankmorales2020/topo-cbp-rlhf-2026)
# ==============================================================================
print("\n" + "=" * 90)
print("[3/4] Loading Topo-CBP Alignment Governor (GPT-OSS-20B Backbone + Certified Heads)...")
print("=" * 90)

REPO_RLHF = "frankmorales2020/topo-cbp-rlhf-2026"
BASE_RLHF_MODEL = "openai/gpt-oss-20b"
HIDDEN_SIZE_RLHF = 2880

class TOPOCBPInferenceModel(nn.Module):
    def __init__(self, base_model: nn.Module):
        super().__init__()
        self.base_model = base_model
        dev = next(base_model.parameters()).device
        self.classifier_A = nn.Linear(HIDDEN_SIZE_RLHF, 2, dtype=torch.bfloat16).to(dev)
        self.classifier_B = nn.Linear(HIDDEN_SIZE_RLHF, 2, dtype=torch.bfloat16).to(dev)
        self.classifier_C = nn.Linear(HIDDEN_SIZE_RLHF, 2, dtype=torch.bfloat16).to(dev)
        self.current_task = "A"

    def switch_task(self, task: str):
        assert task in ("A", "B", "C"), "Task must be one of 'A', 'B', or 'C'"
        self.current_task = task

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor = None) -> torch.Tensor:
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
        hidden_states = outputs.hidden_states[-1]
        if attention_mask is not None:
            seq_lens = torch.eq(attention_mask, 1).int().sum(-1) - 1
            batch_idx = torch.arange(input_ids.shape[0], device=input_ids.device)
            last_hidden = hidden_states[batch_idx, seq_lens, :]
        else:
            last_hidden = hidden_states[:, -1, :]
        head = getattr(self, f"classifier_{self.current_task}")
        if hasattr(head, "linear"):
            return head.linear(last_hidden)
        return head(last_hidden)

with quiet():
    tokenizer_rlhf = AutoTokenizer.from_pretrained(REPO_RLHF, trust_remote_code=True)
    if tokenizer_rlhf.pad_token is None:
        tokenizer_rlhf.pad_token = tokenizer_rlhf.eos_token

    base_rlhf_model = AutoModelForCausalLM.from_pretrained(
        BASE_RLHF_MODEL,
        dtype=torch.bfloat16,
        trust_remote_code=True,
        device_map="auto" if torch.cuda.is_available() else None
    )
    if not torch.cuda.is_available():
        base_rlhf_model.to(DEVICE)

    model_rlhf = TOPOCBPInferenceModel(base_rlhf_model)

    weights_path_rlhf = hf_hub_download(repo_id=REPO_RLHF, filename="topo_cbp_best.pt")
    state_dict_rlhf = torch.load(weights_path_rlhf, map_location="cpu", weights_only=True)

    model_dict = model_rlhf.state_dict()
    filtered_dict = {}
    for k, v in state_dict_rlhf.items():
        norm_key = k.replace(".linear.", ".")
        target_key = k if k in model_dict else (norm_key if norm_key in model_dict else None)
        if target_key is not None and model_dict[target_key].shape == v.shape:
            filtered_dict[target_key] = v.to(device=model_dict[target_key].device)

    model_rlhf.load_state_dict(filtered_dict, strict=False)
    model_rlhf.eval()

    del state_dict_rlhf, filtered_dict
    gc.collect()
    torch.cuda.empty_cache()

RLHF_LABELS = {
    "A": {0: "World", 1: "Sports"},
    "B": {0: "Business", 1: "Sci/Tech"},
    "C": {0: "World", 1: "Sci/Tech"}
}

def predict_rlhf(text: str, task: str = "A"):
    model_rlhf.switch_task(task)
    inputs = tokenizer_rlhf(text, max_length=64, padding="max_length", truncation=True, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        logits = model_rlhf(inputs.input_ids, inputs.attention_mask)
        probs = F.softmax(logits.float(), dim=-1).squeeze().cpu().numpy()
    class_idx = int(probs.argmax())
    return {
        "task": task,
        "label": RLHF_LABELS[task][class_idx],
        "confidence": float(probs.max()),
        "logits": logits.squeeze().tolist()
    }

print("  [✓] Alignment Governor Online.")

# ==============================================================================
# 4. COGNITIVE REASONING AGENT (frankmorales2020/qwen2.5-0.5b-topo-governed-cbp-fineweb)
# ==============================================================================
print("\n" + "=" * 90)
print("[4/4] Loading Cognitive Reasoning Agent (Qwen2.5-0.5B-Topo-Governed)...")
print("=" * 90)

REPO_QWEN = "frankmorales2020/qwen2.5-0.5b-topo-governed-cbp-fineweb"
with quiet():
    tokenizer_qwen = AutoTokenizer.from_pretrained(REPO_QWEN)
    model_qwen = AutoModelForCausalLM.from_pretrained(
        REPO_QWEN,
        dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32,
        device_map="auto"
    )
    model_qwen.eval()

print("  [✓] Cognitive Reasoning Agent Online.")

# ==============================================================================
# 5. MULTI-AGENT INFERENCE & DELIBERATION EXECUTION CYCLE
# ==============================================================================
print("\n" + "=" * 90)
print("EXECUTING REAL MULTIMODAL DELIBERATION ACROSS ALL 4 HUBS")
print("=" * 90)

# Step 1: Vision Perception (LIDC-IDRI Lung Nodule Crop via URL)
IMAGE_URL = "https://raw.githubusercontent.com/frank-morales2020/MLxDL/main/LIDC-IDRI-0001_n0.png"
image_path, _ = urllib.request.urlretrieve(IMAGE_URL, "LIDC-IDRI-0001_n0.png")
res_vis = predict_vision("14", image=image_path)
print(f"1. [Vision Agent - Task 14] CT Scan Diagnosis: {res_vis['label'].upper()} (Confidence: {res_vis['confidence']*100:.2f}%)")
print(f"                       Probabilities: {res_vis['probabilities']}")

# Step 2: Embodied World Model (Swish-Gated 3D Dynamics Rollout)
z_t = torch.randn(1, 128, device=DEVICE)
a_t = torch.tensor([[0.25, -0.41, 0.12, 0.05, -0.18, 0.33]], device=DEVICE)
with torch.no_grad():
    z_next = world_model(z_t, a_t)
print(f"\n2. [World Model Agent] Transition (z_t, a_t) -> Next State Norm: {torch.norm(z_next).item():.4f} (Shape: {list(z_next.shape)})")

# Step 3: Alignment Governor Domain Clearance
safety_text = "Autonomous robotic biopsy trajectory approved following non-proliferation safety protocols."
res_gov = predict_rlhf(safety_text, task="B")
print(f"\n3. [Governor Agent]    Safety Policy Clearance (Task B): {res_gov['label']} (Confidence: {res_gov['confidence']*100:.2f}%)")

# Step 4: Cognitive Reasoning Synthesis (Deduplicated Stream Generation)
synthesis_prompt = (
    f"Perception finding: Lung nodule is {res_vis['label']} ({res_vis['confidence']*100:.1f}% confidence).\n"
    f"Physical dynamics: 3D latent state stable (Norm: {torch.norm(z_next).item():.2f}).\n"
    f"Safety governance: Execution cleared under {res_gov['label']} classification.\n"
    f"Prescribe operational trajectory and clinical conclusion:"
)

cog_inputs = tokenizer_qwen(synthesis_prompt, return_tensors="pt").to(DEVICE)
input_len = cog_inputs.input_ids.shape[-1]

with torch.no_grad():
    cog_gen = model_qwen.generate(
        **cog_inputs,
        max_new_tokens=90,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.25,
        no_repeat_ngram_size=3,
        do_sample=True,
        pad_token_id=tokenizer_qwen.eos_token_id
    )

# Slice output to strip the prompt echo and display only the model's new generation
cog_output = tokenizer_qwen.decode(cog_gen[0][input_len:], skip_special_tokens=True)
print(f"\n4. [Cognitive Reasoning Agent - Qwen Synthesis]:\n{cog_output.strip()}")

print("\n" + "=" * 90)
print("DELIBERATION AUDIT: ALL 4 REPOSITORIES EXECUTED END-TO-END ON CUDA")
print("=" * 90)

[*] Initializing Four-Agent Deliberation Council on: cuda

[1/4] Loading Topo-Gemma Vision Model (14 Tasks via Unsloth)...


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.68GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

  [✓] Vision Agent Online: Tasks A, B, C, D, E, F, G, H, I, J, K, L, M, 14

[2/4] Loading Topo-CBP Embodied World Model (3D Latent Dynamics)...


...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/139 [00:00<?, ?B/s]

  [✓] Cognitive Reasoning Agent Online.

EXECUTING REAL MULTIMODAL DELIBERATION ACROSS ALL 4 HUBS
1. [Vision Agent - Task 14] CT Scan Diagnosis: MALIGNANT (Confidence: 99.33%)
                       Probabilities: {'benign': 0.0067, 'malignant': 0.9933}

2. [World Model Agent] Transition (z_t, a_t) -> Next State Norm: 9.9667 (Shape: [1, 128])

3. [Governor Agent]    Safety Policy Clearance (Task B): Sci/Tech (Confidence: 97.19%)

4. [Cognitive Reasoning Agent - Qwen Synthesis]:
Pulmonary nodules are benign, the tumor originates in lung parenchyma; pulmonary metastasis of cancer cells can be seen with contrast-enhanced computed tomography or magnetic resonance imaging if present on chest x-ray. The patient has a history of smoking for over twenty years but no evidence of chronic obstructive airway disease (COPA) such as wheezing or cough to suggest COPDA pathology exists at this time because the only findings

DELIBERATION AUDIT: ALL 4 REPOSITORIES EXECUTED END-TO-END ON CUDA


In [3]:
!nvidia-smi

Thu Oct  1 01:38:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   35C    P0             86W /  600W |   52859MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----